# EDA

This notebook loads the Coffee Invoice dataset, computes the exploratory summary, and writes the markdown report into the same folder.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for _ in range(4):
    if (ROOT / 'backend').exists():
        break
    ROOT = ROOT.parent

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

from app.services.data_service import get_eda_summary, load_enriched_data

df = load_enriched_data()
summary = get_eda_summary(df)
summary


{'rows': 20000,
 'date_start': '2023-01-01',
 'date_end': '2023-12-31',
 'unique_stores': 45,
 'shift_distribution': {'Night': 1510,
  'Morning': 9806,
  'Afternoon': 5982,
  'Evening': 2702},
 'holiday_share_pct': 1.68,
 'top_products': [{'product': 'Coffee', 'count': 9216},
  {'product': 'Tea', 'count': 4963},
  {'product': 'Pastry', 'count': 2524},
  {'product': 'Sandwich', 'count': 1781},
  {'product': 'Smoothie', 'count': 978}],
 'top_cities': [{'city': 'New York', 'count': 2288},
  {'city': 'Vancouver', 'count': 2287},
  {'city': 'Toronto', 'count': 2284},
  {'city': 'London', 'count': 2251},
  {'city': 'Sydney', 'count': 2211}],
 'day_of_week_distribution': {'Monday': 2061,
  'Tuesday': 2347,
  'Wednesday': 2448,
  'Thursday': 2843,
  'Friday': 3029,
  'Saturday': 3925,
  'Sunday': 3347},
 'sales_summary': {'quantity_mean': 1.71,
  'quantity_max': 9,
  'unit_price_mean': 4.05,
  'unit_price_max': 18.75}}

In [2]:
def build_markdown(summary: dict) -> str:
    top_products = "\n".join(
        f"- {item['product']}: {item['count']} records"
        for item in summary.get('top_products', [])
    ) or '- No product category summary available.'
    top_cities = "\n".join(
        f"- {item['city']}: {item['count']} records"
        for item in summary.get('top_cities', [])
    ) or '- No city summary available.'

    lines = [
        '# EDA Report',
        '',
        '## Dataset overview',
        f"- Rows: {summary['rows']}",
        f"- Date range: {summary['date_start']} to {summary['date_end']}",
        f"- Unique stores: {summary['unique_stores']}",
        f"- Holiday share: {summary['holiday_share_pct']}%",
        '',
        '## Shift distribution',
    ]
    for shift_name, count in summary['shift_distribution'].items():
        lines.append(f'- {shift_name}: {count}')

    lines.extend([
        '',
        '## Day-of-week distribution',
    ])
    for day_name, count in summary['day_of_week_distribution'].items():
        lines.append(f'- {day_name}: {count}')

    lines.extend([
        '',
        '## Top products',
        top_products,
        '',
        '## Top cities',
        top_cities,
        '',
        '## Sales heuristics',
    ])
    for key, value in summary.get('sales_summary', {}).items():
        lines.append(f'- {key}: {value}')
    if not summary.get('sales_summary'):
        lines.append('- No numeric sales metrics were available for this dataset.')
    return '\n'.join(lines) + '\n'

output_path = ROOT / 'notebooks' / 'exploratory' / 'EDA_REPORT.md'
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(build_markdown(summary), encoding='utf-8')
print(f'EDA report saved to {output_path}')


EDA report saved to D:\MayHoc\CoffeeInvoicePrediction\notebooks\exploratory\EDA_REPORT.md
